## Data preprocessing

In [2]:
import os
import numpy as np
import cv2
from glob import glob
from tqdm import tqdm
import imageio.v2 as imageio
from albumentations import HorizontalFlip, VerticalFlip, Rotate

/Users/soumyajit/Documents/BioMed/Unets/.venv/lib/python3.12/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>
  data = fetch_version_info()


In [4]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)


def load_data(path):
    train_x = sorted(glob(os.path.join(path, "train", "images", "*.jpg")))
    train_y = sorted(glob(os.path.join(path, "train", "masks", "*.jpg")))

    test_x = sorted(glob(os.path.join(path, "test", "images", "*.jpg")))
    test_y = sorted(glob(os.path.join(path, "test", "masks", "*.jpg")))

    val_x = sorted(glob(os.path.join(path, "valid", "images", "*.jpg")))
    val_y = sorted(glob(os.path.join(path, "valid", "masks", "*.jpg")))

    return (train_x, train_y), (test_x, test_y), (val_x, val_y)


def augumentation(images, masks, save_path, augment=True):
    size = (256, 256)

    for idx, (x, y) in tqdm(enumerate(zip(images, masks)), total=len(images)):
        """Extracting the name of the file"""
        name = x.split("/")[-1].split(".")[0]

        """Reading image and mask"""
        x = cv2.imread(x, cv2.IMREAD_COLOR)
        y = imageio.imread(y)

        if augment == True:

            aug = HorizontalFlip(p=1.0)
            augmented = aug(image=x, mask=y)
            x1 = augmented["image"]
            y1 = augmented["mask"]

            aug = VerticalFlip(p=1.0)
            augmented = aug(image=x, mask=y)
            x2 = augmented["image"]
            y2 = augmented["mask"]

            aug = Rotate(limit=45, p=1.0)
            augmented = aug(image=x, mask=y)
            x3 = augmented["image"]
            y3 = augmented["mask"]

            X = [x1, x2, x3]
            Y = [y1, y2, y3]

        else:
            X = [x]
            Y = [y]

        index = 0
        for i, m in zip(X, Y):
            i = cv2.resize(i, size)
            m = cv2.resize(m, size)
            tmp_image_name = f"{name}_{index}.png"
            tmp_mask_name = f"{name}_{index}.png"

            image_path = os.path.join(save_path, "image", tmp_image_name)
            mask_path = os.path.join(save_path, "mask", tmp_mask_name)

            cv2.imwrite(image_path, i)
            cv2.imwrite(mask_path, m)

            index += 1

In [5]:
# """Seeding"""
# np.random.seed(42)

# """Load the data"""
# data_path = "../data/TNBC-ImageMask-Dataset-V1-2"
# if not os.path.isdir(data_path):
#     raise FileNotFoundError(f"Dataset directory not found: {data_path}")
# (train_x, train_y), (test_x, test_y), (val_x, val_y) = load_data(data_path)

# print(f"Train: {len(train_x)} - {len(train_y)}")
# print(f"Test: {len(test_x)} - {len(test_y)}")
# print(f"Test: {len(val_x)} - {len(val_y)}")

# """Create Directories to sava augumented data"""
# create_dir("../dataset/train/image")
# create_dir("../dataset/train/mask")
# create_dir("../dataset/test/image")
# create_dir("../dataset/test/mask")
# create_dir("../dataset/val/image")
# create_dir("../dataset/val/mask")


# """Data Augumetation"""
# augumentation(train_x, train_y, "../dataset/train/", augment=True)
# augumentation(test_x, test_y, "../dataset/test/", augment=False)
# augumentation(val_x, val_y, "../dataset/val/", augment=False)

Train: 880 - 880
Test: 55 - 55
Test: 165 - 165


100%|████████████████████████████████████████| 165/165 [00:01<00:00, 142.03it/s]


## Training

In [3]:
import os
import random
import numpy as np
import torch
from glob import glob
import time

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from  segmentation_models_pytorch import Unet
import numpy as np
import pandas as pd


from classes import Model_Training as MT
from classes import DataDrive as DD
from classes import loss_functions as LF
from classes import Unet as U

In [4]:
""" Seeding the randomness. """
def seeding(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

""" Create a directory. """
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

""" Calculate the time taken """
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [5]:
""" seeding"""
seeding(42)

"""Directories"""
create_dir("../files")
create_dir("../train_data")
data_root = "../dataset"

"""Loading Data"""

train_x = sorted(glob(f"{data_root}/train/images/*"))
train_y = sorted(glob(f"{data_root}/train/masks/*"))

valid_x = sorted(glob(f"{data_root}/val/images/*"))
valid_y = sorted(glob(f"{data_root}/val/masks/*"))

if len(train_x) == 0 or len(valid_x) == 0:
    raise RuntimeError(f"No training data found under {data_root}. Run the augmentation/export cells first.")

data_ = f"Dataset size:\nTrain: {len(train_x)} - Valid: {len(valid_x)}\n"
print(data_)

"""Hyper parameters"""
size = (512, 512)
batch_size = 2
epochs = 10
learning_rate = 1e-4

"""Boolean to set whether the encder is trained or not"""
pre_trained = False

""" Dataset and loader"""

train_data = DD.DataDrive(train_x, train_y)
valid_data = DD.DataDrive(valid_x, valid_y)

train_loader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(dataset=valid_data, batch_size=batch_size, shuffle=False)

#encoders = [ "densenet121", "densenet201"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Dataset size:
Train: 880 - Valid: 165

Using device: mps


In [11]:
import cv2

# Load the image
image = cv2.imread(f'{data_root}/train/images/1001.jpg')

image.shape

(640, 640, 3)

In [ ]:
encoders = ["None"]

for encoder_name in encoders:
    if pre_trained and encoder_name != "None":
        model = Unet(encoder_name, encoder_weights="imagenet", classes=1, activation=None)
        model_name = "Unet_" +"pre_trained_"+ encoder_name
    elif pre_trained == False and encoder_name != "None":
        model = Unet(encoder_name, encoder_weights=None, classes=1, activation=None)
        model_name = "Unet_"+ encoder_name
    else:
        model = U.Unet()
        model_name = "Unet_"+ encoder_name
    checkpoint_path = "../files/" + model_name +".pth"
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5) 
    loss_function = LF.DiceBCELoss()

    """ Training the model """
    best_valid_loss = float("inf")

    losses_values  = np.array(["epoch", "Train", "Test"])

    M = MT.Model_Training(model, train_loader, valid_loader, optimizer, device, loss_function)

    for epoch in range(epochs):
        start_time = time.time()

        train_loss = M.train()
        valid_loss = M.evaluate() 

        """ Saving the model """
        if valid_loss < best_valid_loss:
            data_str = f"Valid loss improved from {best_valid_loss:2.4f} to {valid_loss:2.4f}. Saving checkpoint: {checkpoint_path}"
            print(data_str)

            best_valid_loss = valid_loss
            torch.save(model.state_dict(), checkpoint_path)

        end_time = time.time()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)

        data_str = f'Epoch: {epoch + 1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s\n'
        data_str += f'\tTrain Loss: {train_loss:.3f}\n'
        data_str += f'\t Val. Loss: {valid_loss:.3f}\n'
        losses_values = np.vstack((losses_values, np.array([epoch, train_loss, valid_loss])))
        print(data_str)
    C = pd.Index(["Epoch", "Train", "Valid"], name="columns")
    df = pd.DataFrame(data=losses_values, columns=C)
    df.drop(index=df.index[0], axis=0, inplace=True)
    csv_path = "../train_data/" + model_name + ".csv"
    df.to_csv(csv_path, index=False)